# 01 - LiDAR + calibration visual sanity check (milestone 1)

Goal: SEE the geometry before any ML. By the end you should understand:
- what a point cloud is (a 3D scatter, not a grid)
- projection (LiDAR points -> image pixels) and why calibration matters
- BEV (top-down map; cars are blobs)

Runs on **synthetic data** if KITTI isn't downloaded yet, so you can learn today.
Drop real KITTI into `../data/kitti/` later and re-run.

Run from the `fusion-benchmark/` directory: `jupyter notebook notebooks/01_lidar_visual_sanity_check.ipynb`

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # so `import common...` works
import numpy as np
import matplotlib.pyplot as plt
from common.sensors.calibration import Calib
from common.sensors.projection import lidar_to_image, render_depth_image
from common.geometry.bev import points_to_bev_image
%matplotlib inline

USE_KITTI = os.path.isdir('../data/kitti/image_2')
print('Using KITTI' if USE_KITTI else 'No KITTI found -> using synthetic data (fine for learning)')

In [ ]:
def get_frame():
    if USE_KITTI:
        from data.loaders.paired_loader import KittiPairedDataset
        cfg = {'data_root': '../data/kitti', 'image_size': [384, 1280],
               'lidar': {'max_points': 15000}, 'classes': ['Car']}
        s = KittiPairedDataset(cfg, 'train')[0]
        img = s['image'].permute(1,2,0).numpy()
        pts = s['points'].numpy()
        calib = s['calib']
    else:
        from data.loaders.synthetic_loader import SyntheticPairedDataset
        cfg = {'image_size': [384, 1280], 'lidar': {'max_points': 15000}}
        s = SyntheticPairedDataset(cfg, 'val', length=8)[0]
        img = s['image'].permute(1,2,0).numpy()
        pts = s['points'].numpy()
        calib = s['calib']
    return img, pts, calib

img, pts, calib = get_frame()
H, W = img.shape[:2]
print(f'image {img.shape}, points {pts.shape}, calib P2 fx={calib.P2[0,0]:.1f}')

## 1. The raw inputs: an image + a point cloud
The camera gives a 2D grid of color. The LiDAR gives a LIST of 3D points `(x,y,z,intensity)`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(img); ax[0].set_title('Camera image (2D grid of color)'); ax[0].axis('off')
sub = pts[:3000] if len(pts) > 3000 else pts
sc = ax[1].scatter(sub[:,0], sub[:,2], c=sub[:,2], s=1, cmap='viridis')  # x vs z (top-down-ish)
ax[1].set_xlabel('x (right)'); ax[1].set_ylabel('z (forward)'); ax[1].set_title('LiDAR points (a 3D list, not a grid)')
plt.colorbar(sc, ax=ax[1], label='z (forward, m)'); plt.show()

## 2. Projection: LiDAR -> image pixels (the calibration muscle)
Each 3D point is mapped to a pixel using `P2 @ R0_rect @ Tr_velo_to_cam`. If calibration is right, points land ON objects in the image. **This is the #1 thing to verify before any training.**

In [ ]:
uv, depth, valid = lidar_to_image(pts, calib, H, W)
uv, depth = uv[valid], depth[valid]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(img); ax[0].set_title('image only'); ax[0].axis('off')
ax[1].imshow(img)
sc = ax[1].scatter(uv[:,0], uv[:,1], c=depth, s=1, cmap='jet', vmin=0, vmax=40)
ax[1].set_title('LiDAR overlaid, colored by depth (m)'); ax[1].axis('off')
plt.colorbar(sc, ax=ax[1], label='depth (m)'); plt.show()
print('GATE: do the colored dots sit on cars/objects? If they float off, calibration is wrong.')

## 3. Depth as a 4th channel (early fusion, variant A)
Render the projected points into a dense `(H,W)` depth image. Stack with RGB -> 4-channel input. Empty pixels (no LiDAR) are black -- in early fusion you must mask/interpolate these, not feed zeros silently.

In [ ]:
dimg = render_depth_image(pts, calib, H, W)
four = np.dstack([img, dimg / 40.0])  # RGB + normalized depth
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(dimg, cmap='gray', vmin=0, vmax=40); ax[0].set_title('depth channel (m)'); ax[0].axis('off')
ax[1].imshow(four[..., :3]); ax[1].set_title('RGB (depth channel is the 4th, not shown here)'); ax[1].axis('off')
plt.show()
print(f'fraction of pixels with LiDAR depth: {(dimg>0).mean():.3f}  (sparse -> mask it!)')

## 4. BEV: the top-down map (variant D lives here)
Drop the points onto a top-down grid (forward x right). Cars become compact blobs; rotation is just an angle. This is why 3D detection works in BEV.

In [ ]:
# points need to be in cam frame for points_to_bev_image (x right, y down, z forward)
cam_pts = calib.velo_to_cam(pts[:, :3])
cam_pts = np.hstack([cam_pts, pts[:, 3:4]])  # keep intensity
bev = points_to_bev_image(cam_pts, range_m=32.0, res=0.2)
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(bev, origin='lower'); ax.set_title('BEV top-down [height, intensity, density]'); ax.axis('off')
plt.show()
print('GATE: can you see car-shaped blobs ahead (top of image = forward)?')

## 5. Pillars (the PointPillars atom)
A pillar = all points in one BEV ground cell, stacked vertically. PointPillars runs a tiny MLP per point, max-pools per pillar, scatters back -> a 2D pseudo-image a normal CNN can eat.

In [ ]:
range_m, res = 32.0, 0.2
r = ((cam_pts[:,2] + range_m) / res).astype(int)
c = ((cam_pts[:,0] + range_m) / res).astype(int)
m = (r>=0)&(r<320)&(c>=0)&(c<320)
counts = np.zeros((320,320))
np.add.at(counts, (r[m], c[m]), 1)
fig, ax = plt.subplots(1, 1, figsize=(6,6))
ax.imshow(counts.T, origin='lower', cmap='magma'); ax.set_title('points per pillar (density)'); ax.axis('off')
plt.show()
print(f'occupied pillars: {(counts>0).sum()} / {320*320}  (very sparse -> scatter, not dense conv)')